# 06 — Experiments: opportunity ranking and its evaluation

**Project:** Property Investment Intelligence System (BIU DS23)
**Owner:** Idan Gilad — models and quantitative evaluation
**Status:** Development. `USE_SYNTHETIC = True` runs end to end on synthetic data; `False` runs on the real city.

## Purpose

Answer the question *"what does the system rank?"* (README, **Opportunity definition and score**), then check whether the ranking can be trusted.

**An opportunity is a combination of a neighbourhood, a room type and a number of bedrooms, and its score is calculated from three defined components:**

| # | Component | Question | Source |
| --- | --- | --- | --- |
| 1 | Expected nightly price | How much will such a property earn per booked night? | the selected price model, **out-of-fold** for every listing |
| 2 | Expected booked nights | How often will it be booked? | stated approximation from the review rate (Inside Airbnb's occupancy method) |
| 3 | Price certainty | How reliable is component 1? | lower end of the 80% conformal range / expected price |

**Score (currency/year) = booked nights × expected nightly price × price certainty**, a conservative annual rental revenue.

## Inputs and outputs

| | Location |
| --- | --- |
| Input | `data/processed/listings_features.parquet`, the model selected by `05_model_evaluation` (or `04_modeling`'s development pick), its interval model `cqr_<city>_<version>.joblib`, the split manifest |
| Output | `reports/<city>/opportunities.csv` (one row per opportunity), `opportunity_listings.csv` (the evidence, one row per listing), `opportunities.json` (how the table was made). Read by the agent's tool `src/models/opportunity_tool.py` |

## Guardrails

- The score is **rental revenue potential**, not return on investment: there are no purchase prices or costs in the data.
- Booked nights are an **approximation from reviews**, with stated assumptions, not observed bookings.
- **The one-shot rule of `05_model_evaluation` is respected.** Until 05 has made its confirmatory test run for this city and split, the test listings are left out of the table (label `PROVISIONAL`). Afterwards, all priced listings are used (label `FINAL`). Nothing here feeds back into model selection.
- The scoring logic lives in `src/models/opportunities.py` (tested in `tests/test_opportunities.py`), so the agent's tool and this notebook use exactly the same definitions.

## 1. Setup

In [ ]:
from pathlib import Path
from dataclasses import asdict
import json
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent                     # the notebook lives in notebooks/
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.models.make_split import load_listings, ensure_snapshot_date, select_city, load_manifest
from src.models.data import prepare_listings, make_synthetic_listings, ID_COL, SNAPSHOT_COL, TARGET
from src.models.feature_audit import load_audit, apply_row_transforms
from src.models.target_policy import apply_target_policy
from src.models.cities import city_slug, city_data_sha256
from src.models.opportunities import (OpportunitySettings, load_model_bundle, confirmatory_run_done,
                                      cross_fitted_predictions, listing_components, oof_coverage,
                                      opportunity_table, unknown_band_count, save_opportunities,
                                      segment_price_check, occupancy_agreement, bootstrap_stability,
                                      assumption_sensitivity, forward_check)
from src.models.opportunity_tool import OpportunityTool

# ---- Data source switch ----------------------------------------------------
USE_SYNTHETIC = False          # True: synthetic listings and the synthetic models of 04_modeling
CITY = "dummy_city" if USE_SYNTHETIC else "Amsterdam"
CITY_COL = "location"
RANDOM_SEED = 42
N_FOLDS = 5                    # out-of-fold predictions (components 1 and 3)

# ---- The score's assumptions (README; agree them as a team) -------------------
SETTINGS = OpportunitySettings(
    min_listings=20,           # priced listings needed to rank an opportunity
    review_rate=0.5,           # share of stays that leave a review
    avg_stay_nights=3.0,       # assumed stay; the listing's minimum stay if longer
    occupancy_cap=0.7,         # booked nights capped at 70% of the year
)
if USE_SYNTHETIC:              # 1,500 synthetic listings spread over ~80 segments
    SETTINGS = SETTINGS.with_(min_listings=10)

# ---- Ranking checks -----------------------------------------------------------
TOP_N = 10                     # "the top 10 opportunities"
N_BOOT = 200                   # bootstrap resamples for the stability check

REPORTS_DIR = REPO_ROOT / "reports" / ("synthetic_demo" if USE_SYNTHETIC else city_slug(CITY))
BAR_COLOR = "#2a78d6"
pd.set_option("display.max_columns", 30)
print(f"city: {CITY}   reports: {REPORTS_DIR.relative_to(REPO_ROOT)}   settings: {SETTINGS.describe()['occupancy']}")

In [ ]:
# The model 05_model_evaluation selected for this city (before that: 04's development pick)
pipe, cqr, meta, MODEL_CHOICE = load_model_bundle(REPO_ROOT, CITY, synthetic=USE_SYNTHETIC)
FEATURES = meta["features"]
CURRENCY = meta["currency"]
MODEL_VERSION = f"{meta['model']}_{meta['city_slug']}_{meta['version']}"
print(f"model: {MODEL_VERSION} ({MODEL_CHOICE})")
print(f"interval model: 80% CQR, Q = {cqr['q']:+.4f}; split {meta['split_version']}; {len(FEATURES)} features")

## 2. Listings: the evidence for each opportunity

The same preparation as `04_modeling`: parse prices and bathrooms, apply the audit's row transforms, then the **target policy frozen in the model's metadata** (price cap, no long-term rentals). Only the **latest snapshot** is used: the current market.

In [ ]:
if USE_SYNTHETIC:
    raw = make_synthetic_listings(seed=RANDOM_SEED)
    # The synthetic table has no 12-month review count; derive one for the demo only
    rng = np.random.default_rng(RANDOM_SEED)
    raw["number_of_reviews_ltm"] = rng.binomial(raw["number_of_reviews"].to_numpy(), 0.6)
else:
    raw = ensure_snapshot_date(load_listings(REPO_ROOT / "data" / "processed" / meta["data_source"]))
    raw = select_city(raw, CITY, CITY_COL)
    if meta.get("data_sha256") and city_data_sha256(raw) != meta["data_sha256"]:
        warnings.warn("the listings differ from those the model was trained on (new or reprocessed data)")

df = prepare_listings(raw)                                   # drops listings without a price
df = apply_row_transforms(df, load_audit(REPO_ROOT / "evaluation" / "feature_audit.csv"))
df, removed = apply_target_policy(df, meta["target_policy"])
print(f"removed by the frozen target policy: {removed.to_dict()}")

snapshots = sorted(df[SNAPSHOT_COL].unique())
SNAPSHOT = pd.Timestamp(snapshots[-1])
manifest = load_manifest(REPO_ROOT / "evaluation" / "splits" / meta["split_file"])
test_ids = set(manifest["test"])

# One-shot rule: test listings join the table only after 05's confirmatory run
TEST_DONE = USE_SYNTHETIC or confirmatory_run_done(REPO_ROOT, CITY, meta["split_version"])
if USE_SYNTHETIC:
    LABEL = "SYNTHETIC"
elif TEST_DONE and MODEL_CHOICE.startswith("selected"):
    LABEL = "FINAL"
else:
    LABEL = "PROVISIONAL"


def evidence_rows(frame):
    # the listings allowed into the table at this stage of the project
    return frame if TEST_DONE else frame[~frame[ID_COL].isin(test_ids)]


current = evidence_rows(df[df[SNAPSHOT_COL] == SNAPSHOT]).reset_index(drop=True)
print(f"snapshot {SNAPSHOT:%Y-%m-%d}: {len(current):,} priced listings used   label: {LABEL}")
if not TEST_DONE:
    print(f"  {int(df[SNAPSHOT_COL].eq(SNAPSHOT).sum() - len(current)):,} test listings left out until "
          "05_model_evaluation has made its confirmatory run")

## 3. Components 1 and 3: out-of-fold price and price range

Every listing gets an expected price and an 80% range from models that **did not see its price**: the listings are split into 5 folds (grouped by listing), and each fold is predicted by fresh copies of the selected model and the two quantile models, trained on the other four. The copies keep the saved models' settings (including the kNN *k*); only the training rows change.

Without this, the listings the model trained on would look "fairly priced" almost by construction, and the ranking would favour whatever the model happened to memorise.

**Check:** the out-of-fold 80% ranges should contain about 80% of asking prices. The conformal widening `Q` comes from `04_modeling`'s calibration, so this confirms it still fits.

In [ ]:
oof = cross_fitted_predictions(pipe, cqr, current[FEATURES], current[TARGET], current[ID_COL],
                               n_folds=N_FOLDS, seed=RANDOM_SEED, log_target=meta.get("log_target", True))
lf = listing_components(current, oof, SETTINGS)          # one row per listing
cov = oof_coverage(lf)
mae_oof = float((lf["asking_price"] - lf["expected_price"]).abs().mean())
print(f"out-of-fold MAE {mae_oof:,.1f} {CURRENCY}/night (listing level, all folds)")
print(f"out-of-fold 80% range: coverage {cov['coverage']:.1%} (target ~80%), "
      f"median width {cov['median_width']:,.0f} {CURRENCY}/night")
lf["position"].value_counts(dropna=False).rename("listings")

## 4. Component 2: expected booked nights

Inside Airbnb's own occupancy model, with its assumptions written down in `SETTINGS`:

> booked nights = reviews in the last 12 months ÷ review rate × max(average stay, minimum stay), capped at 70% of the year

Listings with no reviews in the last 12 months count as **0 nights**: an opportunity full of inactive listings scores low. This is conservative; the share of active listings is shown next to each opportunity.

In [ ]:
nights = lf["booked_nights"]
print(f"booked nights per listing: median {nights.median():.0f}, "
      f"share with 0 (no reviews in 12 months) {(nights == 0).mean():.1%}, "
      f"share at the cap ({SETTINGS.occupancy_cap:.0%} = {SETTINGS.occupancy_cap * 365:.0f} nights) "
      f"{(nights >= SETTINGS.occupancy_cap * 365 - 1e-9).mean():.1%}")
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(nights.dropna(), bins=40, color=BAR_COLOR)
ax.set(title="Expected booked nights per listing (review-rate approximation)",
       xlabel="nights per year", ylabel="listings")
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
plt.show()

## 5. The opportunity table

Listings are grouped into opportunities (neighbourhood × room type × bedroom band). Each opportunity's price and certainty are **medians** over its listings and its booked nights the **mean** (inactive listings count as 0 nights, so a median would turn many opportunities into an exact 0); opportunities with fewer than `min_listings` priced listings get `insufficient_evidence` and no rank. Listings with an unknown bedroom count are not assigned to an opportunity.

In [ ]:
table = opportunity_table(lf, SETTINGS)
ranked = table[table["status"] == "ok"]
print(f"{len(table)} opportunities with listings; {len(ranked)} ranked (at least {SETTINGS.min_listings} listings); "
      f"{len(table) - len(ranked)} insufficient evidence; {unknown_band_count(lf):,} listings with unknown bedrooms")
print(f"ranked opportunities cover {int(ranked['n_listings'].sum()):,} of {len(lf):,} listings")

show = {"rank": "rank", "neighbourhood_cleansed": "neighbourhood", "room_type": "room type",
        "bedroom_band": "bedrooms", "n_listings": "listings",
        "expected_price": f"1. price ({CURRENCY}/night)", "booked_nights": "2. booked nights",
        "price_certainty": "3. certainty", "score": f"score ({CURRENCY}/year)",
        "expected_annual_revenue": f"revenue, no discount ({CURRENCY}/year)",
        "share_active": "share active", "n_below_market": "listings below market"}
ranked[list(show)].rename(columns=show).head(15).round({f"1. price ({CURRENCY}/night)": 0, "2. booked nights": 0,
                                                        "3. certainty": 2, f"score ({CURRENCY}/year)": 0,
                                                        f"revenue, no discount ({CURRENCY}/year)": 0,
                                                        "share active": 2})

In [ ]:
top = ranked.head(15).iloc[::-1]
labels = [f"{r.neighbourhood_cleansed} · {r.room_type} · {r.bedroom_band} bd" for r in top.itertuples()]
fig, ax = plt.subplots(figsize=(9, 0.42 * len(top) + 1))
ax.barh(labels, top["score"], color=BAR_COLOR, height=0.6)
for y, v in enumerate(top["score"]):
    ax.text(v, y, f" {v:,.0f}", va="center", fontsize=8, color="#52514e")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.set(title=f"Top {len(top)} opportunities: conservative annual rental revenue ({CURRENCY}/year)",
       xlabel=f"score ({CURRENCY}/year) = booked nights × expected price × certainty")
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

## 6. Save the table for the agent and the investor view

In [ ]:
saved = save_opportunities(table, lf, {
    "city": CITY, "currency": CURRENCY, "label": LABEL,
    "snapshot_date": f"{SNAPSHOT:%Y-%m-%d}",
    "model_version": MODEL_VERSION, "model_choice": MODEL_CHOICE,
    "split_version": meta["split_version"], "data_sha256": meta.get("data_sha256"),
    "test_listings_included": bool(TEST_DONE),
    "settings": asdict(SETTINGS), "score": SETTINGS.describe()["score"],
    "n_folds": N_FOLDS, "oof_coverage": cov, "oof_mae": mae_oof,
}, REPORTS_DIR)
print(f"saved opportunities.csv, opportunity_listings.csv, opportunities.json to "
      f"{REPORTS_DIR.relative_to(REPO_ROOT)} (table {saved['table_version']}, {LABEL})")

## 7. Ranking evaluation

The README's three levels. **Component quality** (7.1, 7.2), **ranking quality** (7.3–7.5), and the **investigation task** (7.6). None of these checks touches the test partition's role in `05_model_evaluation`: they evaluate the ranking, not the price model.

### 7.1 Component 1 at opportunity level

The ranking uses each opportunity's **median** expected price. How close is it to the median **asking** price of the same listings? (Out-of-fold, so the model never saw these prices.) A large error for one opportunity means its score rests on a price the model gets wrong.

In [ ]:
price_summary, price_by_opp = segment_price_check(table)
print({k: round(v, 2) if isinstance(v, float) else v for k, v in price_summary.items()})
price_by_opp.head(10).round(1)

### 7.2 Component 2 against Inside Airbnb's own estimate

Inside Airbnb publishes `estimated_occupancy_l365d`, computed with the same family of assumptions. The column is **excluded from the price model** (it is partly caused by the price), but it is a fair reference for component 2. High agreement shows the approximation is implemented consistently; **it does not show that it equals real bookings**, which the data does not contain.

In [ ]:
agree = occupancy_agreement(lf, table)
print({k: round(v, 3) if isinstance(v, float) else v for k, v in agree.items()})

### 7.3 Stability: does the ranking depend on which listings happen to be in the data?

Listings are resampled with replacement (`N_BOOT` times) and the table is rebuilt from the same per-listing components. For each top opportunity: how often it stays in the top 10, and the 5th–95th percentile of its rank. A top-10 that reshuffles completely under resampling is noise, not a finding.

This measures sampling noise in the aggregation; the models are not refitted.

In [ ]:
stab_summary, stab = bootstrap_stability(lf, SETTINGS, n_boot=N_BOOT, top_n=TOP_N, seed=RANDOM_SEED)
print({k: round(v, 3) if isinstance(v, float) else v for k, v in stab_summary.items()})
names = table.set_index("segment_key")[["neighbourhood_cleansed", "room_type", "bedroom_band"]]
stab_view = names.join(stab, how="inner").sort_values("rank")
stab_view.head(15).round(2)

In [ ]:
v = stab_view.head(15).iloc[::-1]
labels = [f"{r.neighbourhood_cleansed} · {r.room_type} · {r.bedroom_band} bd" for r in v.itertuples()]
fig, ax = plt.subplots(figsize=(9, 0.42 * len(v) + 1))
y = np.arange(len(v))
ax.hlines(y, v["rank p05"], v["rank p95"], color=BAR_COLOR, lw=2)
ax.plot(v["rank"], y, "o", color=BAR_COLOR, ms=8)
ax.axvline(TOP_N + 0.5, color="#9a9890", ls="--", lw=1)
ax.text(TOP_N + 0.5, len(v) - 0.4, f" top {TOP_N} ", ha="left", va="bottom", fontsize=8, color="#52514e")
ax.set_yticks(y, labels)
ax.invert_xaxis()
ax.set(title=f"Rank of the top opportunities: point = rank, line = 90% bootstrap range ({N_BOOT} resamples)",
       xlabel="rank (1 = best)")
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

### 7.4 Sensitivity to the occupancy assumptions

The score's assumptions are stated, not estimated, so the ranking is rebuilt with alternatives: review rate 30% / 50% / 70%, average stay 2 / 3 / 4 nights.

- A different **review rate** scales every listing's nights by the same factor. It changes the score's **level**, but the **order** only where the 70% cap binds.
- The **average stay** interacts with minimum stays, so it can reorder opportunities.

A top-10 that changes a lot here is **assumption-driven**, and should be reported that way.

In [ ]:
sens = assumption_sensitivity(lf, SETTINGS, review_rates=(0.3, 0.5, 0.7), avg_stays=(2.0, 3.0, 4.0),
                              caps=(SETTINGS.occupancy_cap,), top_n=TOP_N)
sens.round(3)

### 7.5 Forward check: did the top opportunities keep earning?

The strongest test: rank the opportunities at an **earlier snapshot**, then look at a **later** one. Did the top-ranked opportunities show a higher revenue proxy (median asking price × mean booked nights, both from the later snapshot) than the rest? It needs at least two snapshots of the same city; Amsterdam currently has one (June 2026).

In [ ]:
if len(snapshots) >= 2:
    early_snap, later_snap = pd.Timestamp(snapshots[0]), pd.Timestamp(snapshots[-1])
    early = evidence_rows(df[df[SNAPSHOT_COL] == early_snap]).reset_index(drop=True)
    oof_e = cross_fitted_predictions(pipe, cqr, early[FEATURES], early[TARGET], early[ID_COL],
                                     n_folds=N_FOLDS, seed=RANDOM_SEED,
                                     log_target=meta.get("log_target", True), verbose=False)
    table_early = opportunity_table(listing_components(early, oof_e, SETTINGS), SETTINGS)
    later = evidence_rows(df[df[SNAPSHOT_COL] == later_snap])
    fwd = forward_check(table_early, later, SETTINGS, top_n=TOP_N)
    print(f"ranked at {early_snap:%Y-%m-%d}, outcome at {later_snap:%Y-%m-%d}:")
    print({k: round(v, 3) if isinstance(v, float) else v for k, v in fwd.items()})
    if USE_SYNTHETIC:
        print("(synthetic data: prices follow a fixed formula, so this only shows the code works)")
else:
    fwd = {"available": False, "reason": f"one snapshot only ({SNAPSHOT:%Y-%m-%d}); add a later one"}
    print(fwd["reason"])

### 7.6 The investigation task (time saved)

**Task:** for each of the top 10 opportunities, produce an evidence brief: the score breakdown, the listings behind it (most below market first), review signals, and contradictions, with every number cited by `evidence_id`.

| Variant | How | Measured |
| --- | --- | --- |
| Agent | the agent calls `rank_opportunities` (`top`, `breakdown`, `listings`) and `predict_listing_price` | time per brief, cost per query, groundedness, failed tool calls |
| Deterministic table | the investor reads `opportunities.csv` / `opportunity_listings.csv` | time per brief |
| Manual reference | a team member does the same analysis by hand | time per brief, the reference for completeness |

**Groundedness:** load the tool with `evidence_log=...` (section 8). A claim in a brief is grounded if the `evidence_id` it cites is in the log and the number it quotes matches the logged value. The harness belongs to the agent work (Adir); this notebook provides the table and the tool.

## 8. The agent's tool

`src/models/opportunity_tool.py` reads the saved table; the agent never reads the files itself. Three actions: `top`, `breakdown` and `listings`. Neighbourhood names may be partial (`"De Pijp"`), room types informal (`"entire home"`), bedrooms a number or a band. Answers always have the same keys, never raise, and withhold numbers for opportunities with too few listings.

In [ ]:
tool = OpportunityTool.load(REPO_ROOT, include_synthetic=USE_SYNTHETIC)
best = ranked.iloc[0]
calls = [
    {"action": "top", "city": CITY, "n": 5},
    {"action": "breakdown", "city": CITY, "neighbourhood": best["neighbourhood_cleansed"].split(" - ")[0],
     "room_type": best["room_type"].split("/")[0].lower(), "bedrooms": best["bedroom_band"]},
    {"action": "listings", "city": CITY, "neighbourhood": best["neighbourhood_cleansed"],
     "room_type": best["room_type"], "bedrooms": best["bedroom_band"], "n": 3},
]
for c in calls:
    answer = tool.handle_tool_call(c)
    print(f"\n>>> {c}")
    print(json.dumps({k: answer[k] for k in ["status", "status_reason", "results", "evidence_id"]},
                     indent=1, ensure_ascii=False)[:2500])

## 9. Findings and open decisions

Fill in after the real run:

- [ ] Ranked opportunities and how many listings they cover (section 5)
- [ ] Out-of-fold coverage of the 80% ranges (section 3)
- [ ] Stability: mean top-10 overlap and which top opportunities are fragile (7.3)
- [ ] Sensitivity: does the top-10 depend on the occupancy assumptions? (7.4)
- [ ] Agreement of component 2 with Inside Airbnb's estimate (7.2)

Team decisions (README, *Open decisions for the team*): bedroom bands, minimum listings, occupancy inside or outside the score, the third component (certainty or competition), inactive listings as 0 nights.

- [ ] Re-run with label `FINAL` after `05_model_evaluation`'s confirmatory run
- [ ] Forward check once a later Amsterdam snapshot is added (7.5)
- [ ] Investigation-task harness with the agent (7.6)